# 03 - Parse API responses and publish typed Iceberg tables

Run notebook 02 and stop its Spark session first. Start a fresh Spark-enabled kernel. This notebook reads stored Bronze pages from Iceberg; it does not need the API server to remain running.

Spark parses the nested JSON response, explodes its items array, casts money strings to DECIMAL and writes typed product, customer and order tables. Polaris stores catalog references; SeaweedFS stores Iceberg metadata and data objects. This notebook includes its complete setup code.

In [ ]:
import os
import re
import socket
import json

LAB = "http_rest_lake"
assert re.fullmatch(r"[a-z][a-z0-9_]*", LAB)
API_BASE = os.getenv("REST_API_BASE_URL", "http://spark-master:18717").rstrip("/")
LAKE_ROOT = f"s3://warehouse/iceberg/{LAB}"
print("Namespace:", "polaris." + LAB)

In [ ]:
import os

import socket

REQUIRED_ENV = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "POLARIS_CLIENT_ID", "POLARIS_CLIENT_SECRET"]

def preflight():
    missing = [name for name in REQUIRED_ENV if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Missing existing credential environment names: " + ", ".join(missing))
    import importlib.util
    from pathlib import Path
    if importlib.util.find_spec("pyspark") is None:
        raise RuntimeError("Choose the existing Spark-enabled Python kernel; do not pip install PySpark")
    jars = Path(os.environ.get("SPARK_HOME", "/opt/spark")) / "jars"
    patterns = ["iceberg-spark-runtime-*.jar", "iceberg-aws-bundle-*.jar",
                "hadoop-aws-*.jar", "aws-java-sdk-bundle-*.jar"]
    absent = [pattern for pattern in patterns if not list(jars.glob(pattern))]
    if absent:
        raise RuntimeError("Missing image JARs; stop without rebuilding/installing: " + ", ".join(absent))
    print("PREFLIGHT_PASS: required environment names, PySpark and image JARs present")

def start_spark(app_name):
    preflight()
    from pyspark.sql import SparkSession
    if SparkSession.getActiveSession() is not None:
        raise RuntimeError("Stop the existing Spark session and restart this kernel before setup")
    builder = (SparkSession.builder.appName(app_name).master("spark://spark-master:7077")
        .config("spark.driver.host", os.environ.get("POD_IP") or socket.gethostbyname(socket.gethostname()))
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.cores.max", "1").config("spark.executor.cores", "1")
        .config("spark.driver.memory", "512m").config("spark.executor.memory", "512m").config("spark.sql.shuffle.partitions", "2")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.polaris", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.polaris.type", "rest")
        .config("spark.sql.catalog.polaris.uri", os.environ.get("POLARIS_URI", "http://polaris:8181/api/catalog"))
        .config("spark.sql.catalog.polaris.warehouse", "training")
        .config("spark.sql.catalog.polaris.rest.auth.type", "oauth2")
        .config("spark.sql.catalog.polaris.scope", "PRINCIPAL_ROLE:ALL")
        .config("spark.sql.catalog.polaris.oauth2-server-uri", os.environ.get("POLARIS_URI", "http://polaris:8181/api/catalog").rstrip("/") + "/v1/oauth/tokens")
        .config("spark.sql.catalog.polaris.header.Polaris-Realm", "POLARIS")
        .config("spark.sql.catalog.polaris.rest-metrics-reporting-enabled", "false")
        .config("spark.sql.catalog.polaris.credential", os.environ["POLARIS_CLIENT_ID"] + ":" + os.environ["POLARIS_CLIENT_SECRET"])
        .config("spark.sql.catalog.polaris.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.polaris.s3.endpoint", os.environ.get("S3_ENDPOINT", "http://seaweedfs:8333"))
        .config("spark.sql.catalog.polaris.s3.path-style-access", "true")
        .config("spark.sql.catalog.polaris.s3.access-key-id", os.environ["AWS_ACCESS_KEY_ID"])
        .config("spark.sql.catalog.polaris.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
        .config("spark.sql.catalog.polaris.client.region", "us-east-1")
        .config("spark.hadoop.fs.s3a.endpoint", os.environ.get("S3_ENDPOINT", "http://seaweedfs:8333"))
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
        .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
        .config("spark.redaction.regex", "(?i)secret|password|token|access.?key|credential"))
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    print("SPARK_STARTED", spark.version, spark.sparkContext.applicationId)
    return spark


from pyspark.sql import functions as F
spark = start_spark("rest-" + LAB)
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS polaris.{LAB} LOCATION '{LAKE_ROOT}'")

## Parse the stored HTTP response envelopes

In [ ]:
bronze = spark.table(f"polaris.{LAB}.api_pages")
assert bronze.count() == 7
assert bronze.select("entity","page").distinct().count() == 7
assert bronze.select("dataset_version").distinct().count() == 1
entity_schemas = {
    "products":"product_id:BIGINT, product_name:STRING, category:STRING, list_price:STRING",
    "customers":"customer_id:BIGINT, customer_name:STRING, country:STRING, email:STRING",
    "orders":"order_id:BIGINT, customer_id:BIGINT, product_id:BIGINT, quantity:INT, unit_price:STRING, country:STRING"
}
frames = {}
for entity, schema in entity_schemas.items():
    envelope = "STRUCT<dataset_version:STRING,entity:STRING,page:BIGINT,total:BIGINT,next_page:BIGINT,items:ARRAY<STRUCT<" + schema + ">>>"
    parsed = (bronze.filter(F.col("entity")==entity)
        .withColumn("payload", F.from_json("response_json",envelope, {"mode":"FAILFAST"})))
    assert parsed.filter("payload IS NULL OR payload.items IS NULL").count() == 0
    assert parsed.filter("payload.dataset_version <> dataset_version OR payload.entity <> entity OR payload.page <> page").count() == 0
    frames[entity] = parsed.select(F.explode("payload.items").alias("item")).select("item.*")
products = frames["products"].withColumn("list_price",F.col("list_price").cast("decimal(12,2)"))
customers = frames["customers"]
orders = frames["orders"].withColumn("unit_price",F.col("unit_price").cast("decimal(12,2)"))
for frame, key, count in [(products,"product_id",4),(customers,"customer_id",4),(orders,"order_id",6)]:
    assert frame.count() == count and frame.select(key).distinct().count() == count
    assert frame.filter(F.col(key).isNull()).count() == 0
assert products.filter("list_price IS NULL").count() == 0
assert orders.filter("unit_price IS NULL OR quantity <= 0").count() == 0
assert products.filter("category IS NULL").count() == 1
assert customers.filter("email IS NULL").count() == 1
print("PASS: response envelopes parsed, unique IDs and decimal/null handling verified")

## Write complete typed snapshots

Create each table with its explicit location and source schema, then replace its full contents with one Iceberg commit. Repeating the same extract preserves current rows without appending duplicates; snapshot IDs and file names may change. Keep one pipeline writer active. The tables are committed individually, not as one multi-table transaction.

In [ ]:
for name, frame in [("products",products),("customers",customers),("orders",orders)]:
    frame.createOrReplaceTempView("typed_api_source")
    partition_clause = "PARTITIONED BY (country)" if name == "orders" else ""
    spark.sql(f"""CREATE TABLE IF NOT EXISTS polaris.{LAB}.{name} USING iceberg
        {partition_clause} LOCATION '{LAKE_ROOT}/{name}' TBLPROPERTIES ('format-version'='2')
        AS SELECT * FROM typed_api_source WHERE 1=0""")
    frame.writeTo(f"polaris.{LAB}.{name}").overwrite(F.lit(True))
    target = spark.table(f"polaris.{LAB}.{name}").select(*frame.columns)
    assert frame.exceptAll(target).count() == 0 and target.exceptAll(frame).count() == 0
print("PASS: three typed Iceberg tables exactly match the parsed API records")

## Compare the two HTTP extraction approaches and query the lake

In [ ]:
parallel = spark.table(f"polaris.{LAB}.orders_parallel_http").select(*orders.columns)
assert orders.exceptAll(parallel).count() == 0 and parallel.exceptAll(orders).count() == 0
result = spark.sql(f"""SELECT c.country, count(*) AS orders, sum(o.quantity*o.unit_price) AS revenue
    FROM polaris.{LAB}.orders o
    JOIN polaris.{LAB}.customers c ON o.customer_id=c.customer_id
    JOIN polaris.{LAB}.products p ON o.product_id=p.product_id
    GROUP BY c.country ORDER BY c.country""")
result.show()
assert {r.country:(r.orders,str(r.revenue)) for r in result.collect()} == {
    "IN":(3,"85.00"),"UK":(2,"55.00"),"US":(1,"50.00")}
print("PASS: driver/worker HTTP results agree; six joined orders and revenue 190.00")

## Inspect physical files and record repeat-run fingerprints

In [ ]:
import hashlib
summary = {}
for name in ["api_pages","orders_parallel_http","products","customers","orders"]:
    frame = spark.table(f"polaris.{LAB}.{name}")
    rows = sorted(frame.toJSON().collect())
    summary[name] = {"rows":len(rows),"sha256":hashlib.sha256("\n".join(rows).encode()).hexdigest()}
    files = spark.sql(f"SELECT file_path, record_count FROM polaris.{LAB}.{name}.files")
    paths = [r.file_path for r in files.collect()]
    assert paths and all(path.startswith(LAKE_ROOT + "/" + name + "/") for path in paths)
    files.show(truncate=False)
    spark.sql(f"SELECT snapshot_id,operation FROM polaris.{LAB}.{name}.snapshots").show(truncate=False)
print("PASS RESULT_FINGERPRINT " + json.dumps(summary,sort_keys=True))

## Release Spark
Run this cell before switching Spark notebooks. Keep the separate API kernel alive until HTTP extraction finishes.

In [ ]:
spark.stop()